# 06 — GARCH Volatility Modeling

This notebook estimates GARCH(1,1) volatility on log-returns for each selected stock and saves a report and charts.
It aligns with the project rule to use log-returns for volatility analysis.

In [ ]:
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

ROOT = Path.cwd()
PROC_DIR = ROOT / 'data' / 'processed'
REPORT_DIR = ROOT / 'outputs' / 'reports'
CHART_DIR = ROOT / 'outputs' / 'charts' / 'garch'
for directory in [REPORT_DIR, CHART_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from arch import arch_model
from src.data.preprocessor import SELECTED, NAMES

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

def load_close_series(file_path: Path) -> pd.Series:
    frame = pd.read_csv(file_path, index_col=0, parse_dates=True)
    if 'Close' not in frame.columns:
        raise ValueError(f'Missing Close column in {file_path}')
    return frame['Close'].dropna().sort_index()

rows = []

for ticker in SELECTED:
    train_path = PROC_DIR / f'{ticker}_train_full.csv'
    if not train_path.exists():
        print(f'Skipping {ticker}: processed train file not found')
        continue

    close_series = load_close_series(train_path)
    log_returns = np.log(close_series / close_series.shift(1)).dropna() * 100
    if len(log_returns) < 100:
        print(f'Skipping {ticker}: insufficient return history for GARCH')
        continue

    model = arch_model(
        log_returns,
        mean='Constant',
        vol='GARCH',
        p=1,
        q=1,
        dist='normal',
    )
    result = model.fit(disp='off')
    forecast = result.forecast(horizon=5, reindex=False)
    variance = forecast.variance.values[-1]
    vol_5d = np.sqrt(variance)

    rows.append({
        'Ticker': ticker,
        'Company': NAMES.get(ticker, ticker),
        'AIC': round(float(result.aic), 2),
        'BIC': round(float(result.bic), 2),
        'LogReturn_Mean': round(float(log_returns.mean()), 6),
        'LogReturn_Std': round(float(log_returns.std()), 4),
        'Annualized_Volatility_%': round(float(log_returns.std() * np.sqrt(252)), 4),
        'GARCH_5D_Forecast_Vol_%_Day1': round(float(vol_5d[0]), 4),
        'GARCH_5D_Forecast_Vol_%_Day5': round(float(vol_5d[-1]), 4),
        'N_Obs': len(log_returns),
    })

    fig, ax = plt.subplots(figsize=(12, 4))
    (log_returns.abs().rolling(20).mean()).plot(ax=ax, color='#9467bd', linewidth=1.5)
    ax.set_title(f'{ticker} 20-Day Rolling |Log Return| Mean')
    ax.set_ylabel('Percent')
    ax.set_xlabel('Date')
    fig.tight_layout()
    fig.savefig(CHART_DIR / f'{ticker}_garch_volatility_proxy.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

garch_df = pd.DataFrame(rows).sort_values('Annualized_Volatility_%', ascending=False).reset_index(drop=True)
garch_df.to_csv(REPORT_DIR / 'garch_volatility_report.csv', index=False)

display(garch_df)
print(f'Saved: {REPORT_DIR / "garch_volatility_report.csv"}')